In [1]:
from __future__ import print_function

import os


os.chdir("../")

In [2]:
from hyperopt import STATUS_OK, Trials, tpe
from hyperas.distributions import choice, uniform, loguniform
from hyperas import optim

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, metrics
from tensorflow.keras.datasets import cifar10

from sklearn.model_selection import train_test_split

import numpy as np

from utils import (create_compile_args, create_callbacks_list, 
                hyperas_path, i, best_acc, save_logs, init, 
                load_samples, load_cifar10)

In [3]:
def create_data():
    init()

    feature_train, y_train, feature_val, y_val, feature_test, y_test = load_cifar10(preprocess=False, return_features=True, verbose=0)

    return feature_train, y_train, feature_val, y_val, feature_test, y_test

In [4]:
def create_model(feature_train, y_train, feature_val, y_val, feature_test, y_test):
    def dense_layer(units, actv, use_batch_norm, kernel_init):
        dlayer = models.Sequential()

        dlayer.add(layers.Dropout(dense_dropout))
        dlayer.add(layers.Dense(units, activation=actv if not(use_batch_norm or actv == "prelu") else "linear", 
                           kernel_initializer=kernel_init, use_bias=not use_batch_norm))
        dlayer.add(layers.Activation(actv)) if (use_batch_norm and actv != "prelu") else None
        dlayer.add(layers.PReLU()) if actv == "prelu" else None
        dlayer.add(layers.BatchNormalization()) if use_batch_norm else None

        return dlayer


    global best_acc, i


    hidden_num = {{choice(["one", "two", "three", "four"])}}
    units1 = {{choice([2, 4, 8, 16, 32, 64, 128, 256, 512, 1024])}}
    units2 = int(units1 // {{choice([2, 4, 8])}})
    units3 = int(units2 // {{choice([1, 2, 4])}})
    units4 = int(units3 // {{choice([1, 2, 4])}})
    actv = {{choice(["relu", "elu", "selu", "prelu", "tanh", "sigmoid"])}}
    use_normed_data = {{choice([True, False])}}
    use_batch_norm = {{choice([True, False])}}

    distribution = {{choice(["normal", "uniform"])}}
    initialization = {{choice(["glorot", "he", "lecun"])}}
    kernel_init = f"{initialization}_{distribution}"

    dense_dropout = {{choice([0., 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5])}}

    lr = {{choice([0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1])}}
    momentum = {{choice([0.0, 0.01, 0.03, 0.1, 0.3, 0.5, 0.8, 0.9, 0.99])}}
    weight_decay = {{choice([0.0, 0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1])}}
    opt_choice = {{choice(["sgd", "rmsprop", "adam", "nadam", "adamax", "adamw"])}}


    model = models.Sequential(name="cifar10_dnn_model_00")
    model.add(layers.Input(shape=(2048,)))
    model.add(layers.BatchNormalization()) if use_normed_data else None
    model.add(dense_layer(units1, actv, use_batch_norm, kernel_init))
    
    if hidden_num == "two":
        if units2:
            model.add(dense_layer(units2, actv, use_batch_norm, kernel_init))
    elif hidden_num == "three":
        if units2:
            model.add(dense_layer(units2, actv, use_batch_norm, kernel_init))
        if units3:
            model.add(dense_layer(units3, actv, use_batch_norm, kernel_init))
    elif hidden_num == "four":
        if units2:
            model.add(dense_layer(units2, actv, use_batch_norm, kernel_init))
        if units3:
            model.add(dense_layer(units3, actv, use_batch_norm, kernel_init))
        if units4:
            model.add(dense_layer(units4, actv, use_batch_norm, kernel_init))

    model.add(layers.Dropout(dense_dropout))
    model.add(layers.Dense(10, activation="softmax"))

    sgd = optimizers.SGD(learning_rate=lr, momentum=momentum, decay=weight_decay, nesterov=True)
    rmsprop = optimizers.RMSprop(learning_rate=lr, momentum=momentum, decay=weight_decay)
    adam = optimizers.Adam(learning_rate=lr, decay=weight_decay)
    nadam = optimizers.Nadam(learning_rate=lr, decay=weight_decay)
    adamax = optimizers.Adamax(learning_rate=lr, decay=weight_decay)
    adamw = optimizers.experimental.AdamW(learning_rate=lr, weight_decay=weight_decay)

    if opt_choice == "sgd":
        optimizer = sgd
    elif opt_choice == "rmsprop":
        optimizer = rmsprop
    elif opt_choice == "adam":
        optimizer = adam
    elif opt_choice == "nadam":
        optimizer = nadam
    elif opt_choice == "adamax":
        optimizer = adamax
    elif opt_choice == "adamw":
        optimizer = adamw

    model.compile(**create_compile_args(optimizer=optimizer))


    history = model.fit(
        feature_train, y_train,
        batch_size=128,
        epochs=100,
        validation_data=(feature_val, y_val),
        callbacks=create_callbacks_list(min_delta=1e-2, verbose=0),
        verbose=0,
    ).history

    val_acc = metrics.SparseCategoricalAccuracy()(y_val, model.predict(feature_val, verbose=0))

    if val_acc > best_acc:
        best_acc = val_acc
        model.save(os.path.join(hyperas_path, f"{model.name}.h5"))

    save_logs(model.name, i, val_acc, best_acc, 
            search_space=[hidden_num, units1, units2, units3, units4, actv, use_normed_data, 
                        use_batch_norm, distribution, initialization, kernel_init, dense_dropout, 
                        lr, momentum, weight_decay, opt_choice], 
            names=["hidden_num", "units1", "units2", "units3", "units4", "actv", "use_normed_data", 
                "use_batch_norm", "distribution", "initialization", "kernel_init", "dense_dropout", 
                "lr", "momentum", "weight_decay", "opt_choice"], 
            where_to="file")

    i += 1


    return {"loss": -val_acc, "status": STATUS_OK, "model": None}

In [5]:
best_run, best_model = optim.minimize(
    model=create_model,
    data=create_data,
    algo=tpe.suggest,
    max_evals=200,
    trials=Trials(),
    notebook_name="notebooks/cifar10 hp-tuning dnn model"
)

>>> Imports:
#coding=utf-8

from __future__ import print_function

try:
    import os
except:
    pass

try:
    from hyperopt import STATUS_OK, Trials, tpe
except:
    pass

try:
    from hyperas.distributions import choice, uniform, loguniform
except:
    pass

try:
    from hyperas import optim
except:
    pass

try:
    import tensorflow as tf
except:
    pass

try:
    from tensorflow.keras import layers, models, optimizers, metrics
except:
    pass

try:
    from tensorflow.keras.datasets import cifar10
except:
    pass

try:
    from sklearn.model_selection import train_test_split
except:
    pass

try:
    import numpy as np
except:
    pass

try:
    from utils import create_compile_args, create_callbacks_list, hyperas_path, i, best_acc, save_logs, init, load_samples, load_cifar10
except:
    pass

try:
    from sklearn.metrics import classification_report
except:
    pass

>>> Hyperas search space:

def get_space():
    return {
        'hidden_num': hp.choice('hidden_num', [

In [6]:
best_run

{'actv': 0,
 'dense_dropout': 4,
 'distribution': 1,
 'hidden_num': 1,
 'initialization': 1,
 'lr': 5,
 'momentum': 0,
 'opt_choice': 1,
 'units1': 7,
 'units1_1': 0,
 'units1_2': 0,
 'units1_3': 2,
 'use_normed_data': 0,
 'use_normed_data_1': 0,
 'weight_decay': 4}

In [7]:
from sklearn.metrics import classification_report


*_, feature_test, y_test = create_data()
model = models.load_model(os.path.join(hyperas_path, "cifar10_dnn_model_00.h5"))
model.summary()

preds = model.predict(feature_test)
print(classification_report(y_test, np.argmax(preds, axis=-1), digits=4))

Model: "cifar10_dnn_model_00"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 batch_normalization_353 (Ba  (None, 2048)             8192      
 tchNormalization)                                               
                                                                 
 sequential_336 (Sequential)  (None, 256)              525312    
                                                                 
 sequential_337 (Sequential)  (None, 128)              33280     
                                                                 
 dropout_488 (Dropout)       (None, 128)               0         
                                                                 
 dense_488 (Dense)           (None, 10)                1290      
                                                                 
Total params: 568,074
Trainable params: 563,210
Non-trainable params: 4,864
____________________________________